In [1]:
import os
import sys

if os.getcwd().endswith("notebooks"):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from src.data_loader import load_data
from src.model import get_baseline_models
from src.preprocessing import split_and_scale_data

In [2]:
# Reproduce the same leakage-safe split used in preprocessing.
df = load_data("data/creditcard.csv")
X_train, X_val, X_test, y_train, y_val, y_test, scaler = split_and_scale_data(
    df,
    test_size=0.20,
    validation_size=0.20,
    random_state=42,
)

Successfully loaded dataset


In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "auprc": "average_precision",
}

results = []
for name, model in get_baseline_models().items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )
    results.append({
        "model": name,
        **{metric: scores[f"test_{metric}"].mean() for metric in scoring},
    })

cv_results = pd.DataFrame(results).sort_values("auprc", ascending=False)
cv_results

,model,recall,precision,f1,auprc
4,xgboost,0.820050,0.902338,0.858696,0.841393
5,lightgbm,0.812970,0.905060,0.855823,0.831501
3,class_weighted_random_forest,0.714536,0.944074,0.812983,0.826584
0,class_weighted_logistic,0.904449,0.053387,0.100743,0.738550
1,smote_logistic,0.855075,0.376081,0.521491,0.735547
2,undersampled_logistic,0.862155,0.285858,0.428446,0.700363
